# OpenSky Trino 공항 운항 수집 (Google Colab)

공항의 도착·출발 운항을 OpenSky Trino에서 받아 **분석기(icao-peak-hour.html)용 CSV**로 저장합니다.
설치 없이 브라우저에서 실행됩니다.

**사용 방법:** 아래 칸을 **위에서부터 차례로** 왼쪽 ▶ 버튼을 눌러 실행하세요.

1. **설정** — 사용자명, 공항, 기간 입력
2. **준비** — 라이브러리 설치 (처음 한 번)
3. **미리보기** — 보낼 쿼리 확인 (서버 접속 안 함)
4. **수집** — OpenSky 로그인 후 하루씩 조회
5. **내려받기** — 결과 CSV 저장

> OpenSky "Trino - Historical Data" 가이드의 성능 규칙을 지킵니다: 모든 쿼리에 `day` 파티션 조건, 하루씩 순서대로 조회(동시 실행 없음), 필요한 열만 선택, 받은 날짜는 다시 조회하지 않음.
> 규칙을 반복해서 어기면 계정이 정지되므로, 코드를 고쳐 쓰지 마세요.

> ⚠ Colab은 Google Cloud 서버에서 실행됩니다. OpenSky는 *"AWS and other hyperscalers"*의 접속을 막을 수 있다고 밝히고 있어, 4번 단계에서 접속 오류가 나면 내 PC에서 `fetch_flights.py`로 실행하세요.

## 1. 설정

In [ ]:
#@title 1. 설정 { display-mode: "form" }
OPENSKY_USER = ""  #@param {type:"string"}
AIRPORT = "RKSI"  #@param {type:"string"}
START_DATE = "2025-03-01"  #@param {type:"date"}
END_DATE = "2025-03-03"  #@param {type:"date"}
UTC_OFFSET = 9  #@param {type:"number"}
#@markdown **Google Drive에 진행 상황 저장** (권장: 1년치처럼 오래 걸릴 때 Colab 연결이 끊겨도 이어받기 가능)
SAVE_TO_DRIVE = False  #@param {type:"boolean"}

assert OPENSKY_USER.strip(), "OPENSKY_USER 에 OpenSky 사용자명을 입력하세요."
print(f"사용자 {OPENSKY_USER.strip().lower()} · {AIRPORT.upper()} · 현지 {START_DATE} ~ {END_DATE} (UTC{UTC_OFFSET:+g})")

## 2. 준비 (처음 한 번)

In [ ]:
#@title 2. 준비 { display-mode: "form" }
!pip -q install trino

import csv, datetime as dt, os, time

DAY = 86400
TABLE = "flights_data4"
COLUMNS = ["icao24", "callsign", "firstseen", "lastseen", "estdepartureairport", "estarrivalairport"]
# 가이드 "Known Data Gaps" (UTC)
KNOWN_GAPS = [
    ("2023-01-02 23:00", "2023-01-03 10:00"), ("2023-01-18 11:00", "2023-01-23 07:00"),
    ("2023-06-21 13:00", "2023-06-21 22:00"), ("2023-11-15 06:00", "2023-11-16 08:00"),
    ("2023-11-20 01:00", "2023-11-20 03:00"), ("2023-12-02 08:00", "2023-12-05 03:00"),
    ("2024-05-20 10:00", "2024-05-21 05:00"), ("2026-01-02 19:00", "2026-01-03 18:00"),
    ("2026-01-24 08:00", "2026-01-24 17:00"),
]

def utc_ts(s):
    return int(dt.datetime.strptime(s, "%Y-%m-%d %H:%M").replace(tzinfo=dt.timezone.utc).timestamp())

def local_date_to_utc(date_str, tz_hours):
    d = dt.datetime.strptime(date_str, "%Y-%m-%d").replace(tzinfo=dt.timezone.utc)
    return int(d.timestamp() - tz_hours * 3600)

def plan_days(begin_utc, end_utc):
    # day 파티션 = 운항이 끝난(lastseen) UTC 날짜. 다음 날 끝나는 출발편을 위해 마지막 날 + 1일까지.
    first = begin_utc // DAY * DAY
    last = (end_utc - 1) // DAY * DAY + DAY
    return list(range(first, last + 1, DAY))

def query_day(conn, day, airport):
    sql = (f"SELECT {', '.join(COLUMNS)} FROM {TABLE} WHERE day = {int(day)} "
           f"AND (estarrivalairport = ? OR estdepartureairport = ?)")
    cur = conn.cursor()
    cur.execute(sql, (airport, airport))
    rows = cur.fetchall()
    cur.close()
    return rows

def write_rows(path, rows):
    tmp = path + ".tmp"
    with open(tmp, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(COLUMNS)
        for r in rows:
            w.writerow(["" if v is None else v for v in r])
    os.replace(tmp, path)

def read_rows(path):
    with open(path, newline="", encoding="utf-8") as f:
        return [tuple(r[c] for c in COLUMNS) for r in csv.DictReader(f)]

airport = AIRPORT.strip().upper()
begin = local_date_to_utc(START_DATE, UTC_OFFSET)
end = local_date_to_utc(END_DATE, UTC_OFFSET) + DAY
assert end > begin, "종료일이 시작일보다 빠릅니다."
days = plan_days(begin, end)

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    base = "/content/drive/MyDrive/opensky_trino"
else:
    base = "/content/opensky_trino"
cache_dir = os.path.join(base, "cache", airport)
os.makedirs(cache_dir, exist_ok=True)
out_path = os.path.join(base, f"{airport}_{START_DATE}_{END_DATE}.csv")
print("준비 완료. 저장 위치:", base)

## 3. 미리보기 (서버에 접속하지 않음)

In [ ]:
#@title 3. 미리보기 { display-mode: "form" }
print(f"{airport}: 현지 {START_DATE} ~ {END_DATE} → day 파티션 {len(days)}개를 하루씩 조회")
for a, b in KNOWN_GAPS:
    if utc_ts(a) < end and utc_ts(b) > begin:
        print(f"  ⚠ 알려진 데이터 손실 구간과 겹침 (UTC {a} ~ {b}): 이 시간대는 과소 집계됩니다.")
cached = sum(os.path.exists(os.path.join(cache_dir, f"{dt.datetime.fromtimestamp(d, dt.timezone.utc):%Y-%m-%d}.csv")) for d in days)
print(f"  이미 받은 날: {cached}개 · 새로 조회할 날: {len(days) - cached}개")
for d in days[:3]:
    print(f"  예) WHERE day = {d}  ({dt.datetime.fromtimestamp(d, dt.timezone.utc):%Y-%m-%d})")

## 4. 수집

실행하면 **OpenSky 로그인 링크**가 출력됩니다. 링크를 눌러 로그인하면 자동으로 이어서 진행됩니다.
중간에 멈추거나 연결이 끊기면 **2 → 4번 칸을 다시 실행**하세요. 받은 날은 다시 조회하지 않습니다.

In [ ]:
#@title 4. 수집 { display-mode: "form" }
import trino
from trino.auth import OAuth2Authentication

conn = None
all_rows = []
for i, d in enumerate(days, 1):
    label = f"{dt.datetime.fromtimestamp(d, dt.timezone.utc):%Y-%m-%d}"
    cache = os.path.join(cache_dir, f"{label}.csv")
    if os.path.exists(cache):
        rows = read_rows(cache)
        print(f"[{i}/{len(days)}] {label} 캐시 {len(rows)}건")
    else:
        if conn is None:
            # 가이드의 CLI 설정과 동일: 외부(브라우저) 인증, catalog=minio, schema=osky, 사용자명 소문자
            conn = trino.dbapi.connect(host="trino.opensky-network.org", port=443, http_scheme="https",
                                       user=OPENSKY_USER.strip().lower(), auth=OAuth2Authentication(),
                                       catalog="minio", schema="osky")
        t0 = time.time()
        for attempt in range(3):
            try:
                rows = query_day(conn, d, airport)
                break
            except Exception as e:          # 대기열 초과 등: 기다렸다 재시도
                if attempt == 2:
                    raise
                print(f"  오류: {e} → {30 * (attempt + 1)}초 후 재시도")
                time.sleep(30 * (attempt + 1))
        write_rows(cache, rows)
        print(f"[{i}/{len(days)}] {label} 조회 {len(rows)}건 ({time.time() - t0:.1f}초)")
        time.sleep(1)                       # 한 번에 하나씩, 간격 두고
    all_rows.extend(rows)

# 기간 안의 도착(lastseen)·출발(firstseen)만 남기고 중복 제거
seen, keep = set(), []
for icao24, callsign, fs, ls, dep, arr in all_rows:
    fs = int(fs) if fs not in (None, "") else None
    ls = int(ls) if ls not in (None, "") else None
    is_arr = arr == airport and ls is not None and begin <= ls < end
    is_dep = dep == airport and fs is not None and begin <= fs < end
    if (is_arr or is_dep) and (icao24, fs, ls) not in seen:
        seen.add((icao24, fs, ls))
        keep.append((icao24, (callsign or "").strip(), fs or "", ls or "", dep or "", arr or ""))
keep.sort(key=lambda r: (r[2] or r[3]))
write_rows(out_path, keep)
n_arr = sum(1 for r in keep if r[5] == airport and r[3] != "" and begin <= int(r[3]) < end)
n_dep = sum(1 for r in keep if r[4] == airport and r[2] != "" and begin <= int(r[2]) < end)
print(f"\n완료: {out_path}\n운항 {len(keep)}건 · 도착 {n_arr} · 출발 {n_dep}")

## 5. 내려받기

In [ ]:
#@title 5. 내려받기 { display-mode: "form" }
from google.colab import files
files.download(out_path)
print("내려받은 CSV를 분석기의 데이터 소스 → 'CSV 파일 불러오기'에서 고르세요.")

---
- 비상업적 연구 목적으로만 사용하세요. 논문·웹페이지·발표에는 OpenSky 원 논문을 인용하세요: Schäfer et al., "Bringing Up OpenSky: A Large-scale ADS-B Sensor Network for Research", IPSN 2014, pp. 83-94.
- `estdepartureairport` / `estarrivalairport`는 수신 위치로 추정한 공항이라 실제 실적과 차이가 날 수 있습니다.
- 쿼리가 멈췄으면 Trino UI에서 사용자명으로 찾아 KILL 하세요.